# Leitura dos datasets e reescrita com Gemini

Este notebook carrega os arquivos `.csv` e `.json` da pasta `data`, associa cada dataset a um DataFrame diferente e disponibiliza uma funcao para reescrever noticias com uma personalidade definida via API do Gemini.

In [22]:
from pathlib import Path
import os
import random
import re
import time

import pandas as pd
from dotenv import load_dotenv
from google import genai
from google.genai import types

In [23]:
load_dotenv()

if os.getenv("GEMINI_API_KEY"):
    print("GEMINI_API_KEY carregada do .env com sucesso.")
else:
    print("GEMINI_API_KEY nao encontrada. Verifique o arquivo .env.")

GEMINI_API_KEY carregada do .env com sucesso.


In [24]:
def read_dataset(file_path: Path) -> pd.DataFrame:
    suffix = file_path.suffix.lower()

    if suffix == ".csv":
        return pd.read_csv(file_path)
    if suffix == ".json":
        return pd.read_json(file_path)

    raise ValueError(f"Formato de arquivo nao suportado: {file_path.name}")


data_dir = Path("data")
dataset_files = sorted(
    [*data_dir.glob("*.csv"), *data_dir.glob("*.json")],
    key=lambda path: path.name.lower(),
)

if not dataset_files:
    raise FileNotFoundError("Nenhum arquivo CSV ou JSON foi encontrado na pasta 'data'.")

dataset_files

[WindowsPath('data/Latest_News.csv'),
 WindowsPath('data/Latest_News.json'),
 WindowsPath('data/Science_&_Technology_News.csv'),
 WindowsPath('data/Science_&_Technology_News.json'),
 WindowsPath('data/World_Politics_News.csv'),
 WindowsPath('data/World_Politics_News.json')]

In [25]:
def to_variable_name(file_path: Path) -> str:
    name = file_path.stem.lower()
    name = re.sub(r"\W+", "_", name)
    name = re.sub(r"^(\d)", r"df_\1", name)
    return name


dataframes = {}

for dataset_file in dataset_files:
    df_name = to_variable_name(dataset_file)
    df = read_dataset(dataset_file)
    dataframes[df_name] = df
    globals()[df_name] = df

list(dataframes.keys())

['latest_news', 'science___technology_news', 'world_politics_news']

In [26]:
for df_name, df in dataframes.items():
    print(f"{df_name}: {df.shape[0]} linhas x {df.shape[1]} colunas")

latest_news: 86560 linhas x 11 colunas
science___technology_news: 5437 linhas x 11 colunas
world_politics_news: 2933 linhas x 11 colunas


In [27]:
TEXT_COLUMN_CANDIDATES = ("full_description", "content", "description", "title")
DETAILED_TEXT_COLUMNS = ("full_description", "content", "description")


def choose_news_text_column(
    df: pd.DataFrame,
    candidates=TEXT_COLUMN_CANDIDATES,
    prefer_detailed_text: bool = True,
) -> str:
    if prefer_detailed_text:
        priority_candidates = [column for column in DETAILED_TEXT_COLUMNS if column in candidates]
    else:
        priority_candidates = list(candidates)

    fallback_candidates = [column for column in candidates if column not in priority_candidates]

    for column in [*priority_candidates, *fallback_candidates]:
        if column not in df.columns:
            continue

        non_empty_count = df[column].fillna("").astype(str).str.strip().ne("").sum()
        if non_empty_count > 0:
            return column

    raise ValueError(
        "Nenhuma coluna de texto foi encontrada. Informe 'text_column' manualmente."
    )


def resolve_row_text(
    row: pd.Series,
    preferred_column: str | None = None,
    candidates=TEXT_COLUMN_CANDIDATES,
    allow_title_fallback: bool = False,
) -> tuple[str, str]:
    ordered_candidates = []
    if preferred_column:
        ordered_candidates.append(preferred_column)

    ordered_candidates.extend(column for column in candidates if column not in ordered_candidates)

    if not allow_title_fallback:
        ordered_candidates = [column for column in ordered_candidates if column != "title"]

    for column in ordered_candidates:
        if column not in row.index:
            continue

        value = row[column]
        text = "" if pd.isna(value) else str(value).strip()
        if text:
            return column, text

    if allow_title_fallback and "title" in row.index:
        title_value = row["title"]
        title_text = "" if pd.isna(title_value) else str(title_value).strip()
        if title_text:
            return "title", title_text

    raise ValueError("A linha nao possui texto utilizavel nas colunas candidatas.")


def generate_rewrite_with_retry(
    client: genai.Client,
    *,
    model: str,
    prompt: str,
    system_instruction: str,
    max_attempts: int = 5,
    base_delay: float = 2.0,
) -> str:
    last_error = None

    for attempt in range(1, max_attempts + 1):
        try:
            response = client.models.generate_content(
                model=model,
                config=types.GenerateContentConfig(
                    system_instruction=system_instruction,
                    temperature=0.8,
                ),
                contents=prompt,
            )

            rewritten_text = (response.text or "").strip()
            if not rewritten_text:
                raise ValueError("A resposta da API veio vazia.")

            return rewritten_text
        except Exception as exc:
            last_error = exc
            error_text = str(exc)
            is_retryable = (
                "429" in error_text
                or "RESOURCE_EXHAUSTED" in error_text
                or "503" in error_text
                or "UNAVAILABLE" in error_text
            )

            if not is_retryable or attempt == max_attempts:
                break

            sleep_for = base_delay * (2 ** (attempt - 1)) + random.uniform(0, 1)
            time.sleep(sleep_for)

    raise last_error


def rewrite_news_with_personality(
    df: pd.DataFrame,
    personality: str,
    text_column: str | None = None,
    title_column: str = "title",
    model: str = "gemini-2.5-flash-lite",
    api_key: str | None = None,
    output_column: str = "rewritten_news",
    max_rows: int | None = None,
    sleep_seconds: float = 0.0,
    retry_attempts: int = 5,
    allow_title_fallback: bool = False,
) -> pd.DataFrame:
    if df.empty:
        raise ValueError("O DataFrame esta vazio.")

    if not personality or not personality.strip():
        raise ValueError("Informe uma personalidade valida.")

    resolved_api_key = api_key or os.getenv("GEMINI_API_KEY")
    if not resolved_api_key:
        raise ValueError(
            "Defina a GEMINI_API_KEY no ambiente ou passe a chave em 'api_key'."
        )

    resolved_text_column = text_column or choose_news_text_column(df)
    if resolved_text_column not in df.columns:
        raise ValueError(f"A coluna '{resolved_text_column}' nao existe no DataFrame.")

    client = genai.Client(api_key=resolved_api_key)
    rewritten_df = df.copy()
    rewritten_df["source_text_column"] = pd.NA
    rewritten_df[output_column] = pd.NA
    rewritten_df["rewrite_status"] = "not_requested"
    rewritten_df["rewrite_error"] = pd.NA

    target_indexes = list(rewritten_df.index)
    if max_rows is not None:
        target_indexes = target_indexes[:max_rows]

    system_instruction = (
        "Voce reescreve noticias conforme a personalidade pedida, mantendo os fatos centrais "
        "e sem inventar informacoes novas."
    )

    for row_index in target_indexes:
        row = rewritten_df.loc[row_index]

        try:
            source_column, original_text = resolve_row_text(
                row=row,
                preferred_column=resolved_text_column,
                allow_title_fallback=allow_title_fallback,
            )
            rewritten_df.at[row_index, "source_text_column"] = source_column
        except ValueError as exc:
            rewritten_df.at[row_index, "rewrite_status"] = "skipped"
            rewritten_df.at[row_index, "rewrite_error"] = str(exc)
            continue

        title = ""
        if title_column in rewritten_df.columns and pd.notna(rewritten_df.at[row_index, title_column]):
            title = str(rewritten_df.at[row_index, title_column]).strip()

        prompt = f"""
Assuma a seguinte personalidade ao reescrever a noticia:
{personality}

Regras:
- Reescreva no mesmo idioma do texto original.
- Preserve os fatos centrais.
- Nao invente dados, numeros, citacoes ou personagens.
- Ajuste tom, vocabulario e estilo para refletir a personalidade pedida.
- Retorne apenas o texto reescrito.

Titulo:
{title or 'Sem titulo'}

Texto original:
{original_text}
""".strip()

        try:
            rewritten_text = generate_rewrite_with_retry(
                client,
                model=model,
                prompt=prompt,
                system_instruction=system_instruction,
                max_attempts=retry_attempts,
            )
            rewritten_df.at[row_index, output_column] = rewritten_text
            rewritten_df.at[row_index, "rewrite_status"] = "success"
            rewritten_df.at[row_index, "rewrite_error"] = pd.NA
        except Exception as exc:
            rewritten_df.at[row_index, "rewrite_status"] = "error"
            rewritten_df.at[row_index, "rewrite_error"] = str(exc)

        if sleep_seconds > 0:
            time.sleep(sleep_seconds)

    return rewritten_df

In [28]:
# Exemplo de uso com um subconjunto para evitar custo alto na API:
science_rewritten_df = rewrite_news_with_personality(
    df=dataframes["science___technology_news"].head(10),
    personality="um jornalista sensacionalista e alarmista",
    max_rows=1,
    sleep_seconds=5.0,
    retry_attempts=2,
    text_column="description",
    model="gemini-2.5-flash-lite",
)



In [29]:
science_rewritten_df[["title", "source_text_column", "rewritten_news", "rewrite_status", "rewrite_error"]].head()

,title,source_text_column,rewritten_news,rewrite_status,rewrite_error
0,OPPO Reno6 5G Recensione: uno smartphone elega...,description,<NA>,error,"404 NOT_FOUND. {'error': {'code': 404, 'messag..."
1,Un artista smonta Xiaomi MIX Fold: il risultat...,<NA>,<NA>,not_requested,<NA>
2,"WHO: Staaten nicht in der Lage, Pandemie bald ...",<NA>,<NA>,not_requested,<NA>
3,Google Pay már az Erste Banknál is,<NA>,<NA>,not_requested,<NA>
4,Antipasto di Black Friday su Amazon: che scont...,<NA>,<NA>,not_requested,<NA>


In [30]:
# Para inspecionar quais colunas estao mais preenchidas:
# {
#     column: dataframes["science___technology_news"][column].fillna("").astype(str).str.strip().ne("").sum()
#     for column in TEXT_COLUMN_CANDIDATES
#     if column in dataframes["science___technology_news"].columns
# }